# 03 · Route split and CAN target statistics

No random frame split. Splits are route-level and are made separately within each comma2k19 vehicle.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

# ============================================================
# Repository
# ============================================================
REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

# ============================================================
# Paths
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"

COMMA_ROOT = DATA_ROOT / "comma2k19"
RAW_ROOT = COMMA_ROOT / "raw"
PROCESSED_ROOT = COMMA_ROOT / "processed" / "v1"

MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# Install project from the existing pyproject.toml
# --no-deps keeps Colab's/DACON's binary stack intact.
# ============================================================
subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q", "--no-deps", "-e", str(REPO),
    ],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository     :", REPO)
print("Branch         :", BRANCH)
print("Commit         :", commit)
print("RAW_ROOT       :", RAW_ROOT)
print("PROCESSED_ROOT :", PROCESSED_ROOT)

In [ ]:
import json
import pandas as pd

from blackbox_detection.stage3.manifest import (
    build_segment_manifest,
    split_routes,
    compute_target_stats,
    save_stats,
)

manifest = build_segment_manifest(PROCESSED_ROOT)
assert len(manifest), "Processed manifest is empty."

train, val = split_routes(
    manifest,
    val_fraction=0.15,
    seed=20260918,
)

train.to_csv(MANIFEST_ROOT / "comma_train.csv", index=False)
val.to_csv(MANIFEST_ROOT / "comma_val_id.csv", index=False)
manifest.to_csv(MANIFEST_ROOT / "comma_all.csv", index=False)

print("all  :", len(manifest), "segments")
print("train:", len(train), "segments / routes", train.route_id.nunique())
print("val  :", len(val), "segments / routes", val.route_id.nunique())

assert set(train.route_id).isdisjoint(set(val.route_id))

display(manifest.groupby("vehicle_id").size().rename("all"))
display(train.groupby("vehicle_id").size().rename("train"))
display(val.groupby("vehicle_id").size().rename("val"))

In [ ]:
stats = compute_target_stats(train, PROCESSED_ROOT)
save_stats(stats, MANIFEST_ROOT / "target_stats.json")
print(json.dumps(stats, indent=2))

In [ ]:
from blackbox_detection.stage3.schema import read_frame_table
import numpy as np
import matplotlib.pyplot as plt

sample_rows = train.sample(min(100, len(train)), random_state=20260918)

frames = pd.concat(
    [
        read_frame_table(PROCESSED_ROOT / r.metadata_relpath)
        for r in sample_rows.itertuples()
    ],
    ignore_index=True,
)

cols = [
    "speed_mps",
    "accel_from_speed_mps2",
    "steering_deg",
    "yaw_rate_rps",
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, cols):
    x = frames[col].replace([np.inf, -np.inf], np.nan).dropna()
    lo, hi = x.quantile([0.005, 0.995])
    ax.hist(x.clip(lo, hi), bins=100)
    ax.set_title(col)

plt.tight_layout()
plt.show()